# FIFA 21 Data Cleaning (SDI Python Module)

Raw FIFA 21 player data: 18,979 players, 77 columns. Goal: turn messy text columns into clean, typed columns that are ready for analysis.

Cleaning steps:
1. Fill nulls
2. Rename and drop columns
3. Convert text columns to numbers (W/F, SM, IR, Hits, Weight, Height, Value, Wage, Release Clause)
4. Split Team & Contract into Team, Start Contract, End Contract, On Loan
5. Remove duplicates
6. Filter the age range
7. Save the clean file

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# low_memory=False stops the mixed types warning on the Hits column
df = pd.read_csv('/content/fifa21_raw_data.csv', low_memory=False)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
pd.set_option('display.max_rows', None)
df.isnull().sum()

## 1. Nulls

In [ ]:
# Loan Date End is the only column with nulls (17,966), because most players are not on loan
# fill only that column so nulls in other columns are never hidden by accident
df_new = df.copy()
df_new['Loan Date End'] = df_new['Loan Date End'].fillna('No Loan')
df_new['Loan Date End'].value_counts().head()

## 2. Rename and drop columns

In [ ]:
df_new = df_new.rename(columns={'↓OVA': 'Overall Rating'})

# photo and player URLs are not useful for analysis
df_new = df_new.drop(columns=['photoUrl', 'playerUrl'])
df_new.columns

## 3. Text columns to numbers

In [ ]:
# W/F, SM and IR are star ratings like '4 ★' or '4★', so keep the first character and make it a number
for col in ['W/F', 'SM', 'IR']:
    df_new[col] = df_new[col].astype(str).str.strip().str[0].astype(int)

df_new[['W/F', 'SM', 'IR']].describe()

In [ ]:
# Hits looks like '\n372' or '\n1.6K', so strip the newlines and turn 'K' into thousands
def to_number(x):
    x = str(x).strip()
    if x.endswith('K'):
        return float(x[:-1]) * 1000
    return float(x)

df_new['Hits'] = df_new['Hits'].apply(to_number)
df_new['Hits'].describe()

In [ ]:
# Weight looks like '159lbs', so remove 'lbs' (works for 2 or 3 digit weights)
df_new['Weight'] = df_new['Weight'].str.replace('lbs', '', regex=False).astype(float)

# Height looks like 5'7", so convert feet and inches to total inches
feet_inches = df_new['Height'].str.extract(r"(\d+)'(\d+)")
df_new['Height'] = feet_inches[0].astype(int) * 12 + feet_inches[1].astype(int)

print('Average weight (lbs):', round(df_new['Weight'].mean(), 1))
print('Average height (inches):', round(df_new['Height'].mean(), 1))

In [ ]:
# Value, Wage and Release Clause look like '€67.5M', '€560K' or '€500', so convert them to plain euro amounts
def euros(x):
    x = str(x).replace('€', '').strip()
    if x.endswith('M'):
        return float(x[:-1]) * 1_000_000
    if x.endswith('K'):
        return float(x[:-1]) * 1_000
    return float(x)

for col in ['Value', 'Wage', 'Release Clause']:
    df_new[col] = df_new[col].apply(euros)

df_new[['Value', 'Wage', 'Release Clause']].describe()

## 4. Team & Contract

In [ ]:
# raw values look like '\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n'
# loan players look different, e.g. 'Team\nJun 30, 2021 On Loan', so they get their own flag
team_contract = df_new['Team & Contract'].str.strip()
parts = team_contract.str.split('\n')

df_new['Team'] = parts.str[0].str.strip()
contract = parts.str[-1].str.strip()

df_new[['Start Contract', 'End Contract']] = contract.str.extract(r'(\d{4})\s*~\s*(\d{4})')
df_new[['Start Contract', 'End Contract']] = df_new[['Start Contract', 'End Contract']].apply(pd.to_numeric).astype('Int64')
df_new['On Loan'] = contract.str.contains('Loan')

df_new = df_new.drop(columns=['Team & Contract'])
df_new[['Team', 'Start Contract', 'End Contract', 'On Loan']].head()

In [ ]:
# check: every player should have a team, and only loan players should be missing contract years
print('Missing team:', (df_new['Team'] == '').sum())
print('No contract years, not on loan:', (df_new['End Contract'].isna() & ~df_new['On Loan']).sum())
print('Players on loan:', df_new['On Loan'].sum())

## 5. Duplicates

In [ ]:
# ID is the unique player number, so check it as well as full duplicate rows
print('Duplicate rows:', df_new.duplicated().sum())
print('Duplicate player IDs:', df_new['ID'].duplicated().sum())

df_new = df_new.drop_duplicates(subset='ID', keep='first')
df_new.shape

## 6. Age range

In [ ]:
# Decision: keep players aged 18 to 39 for the analysis.
# Players under 18 and 40+ are real players, not data errors, so this is a scope choice, not a cleaning fix.
# Print how many rows the decision removes so it is documented.
before = len(df_new)
df_new = df_new[(df_new['Age'] >= 18) & (df_new['Age'] < 40)]
print('Rows removed by age filter:', before - len(df_new))
df_new.shape

## 7. Final check and save

In [ ]:
df_new.info()

In [ ]:
df_new.to_csv('/content/fifa21_clean.csv', index=False)